# States

> Loading and placeholder card components for the card stack viewport. Empty-state rendering is provided by `cjm-fasthtml-app-core`'s `render_empty_state` (V8 anatomy composition helper) &mdash; card-stack consumers wire that helper into their viewport's empty callback rather than card-stack shipping a local empty-state default.

In [ ]:
#| default_exp components.states

In [ ]:
#| export
from typing import Any, Literal

from fasthtml.common import Div, P, Span

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_display.card import card, card_body
from cjm_fasthtml_daisyui.components.feedback.loading import loading, loading_styles, loading_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui

from cjm_fasthtml_design_system.text_tiers import text_tiers

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.borders import border, border_color
from cjm_fasthtml_tailwind.utilities.effects import shadow
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, justify, grow
)
from cjm_fasthtml_tailwind.utilities.layout import visibility
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.utilities.typography import font_size, italic, text_align
from cjm_fasthtml_tailwind.core.base import combine_classes

# Local imports
from cjm_fasthtml_card_stack.core.html_ids import CardStackHtmlIds

## render_placeholder_card

Rendered in viewport slots that fall outside the items list (e.g., before
the first item or after the last). The placeholder card fills the geometric
slot so the focus position stays centered and the visible-card-count
indicator remains accurate.

The label text (`"Beginning"` / `"End"`) is hidden by default via
`visibility.invisible` &mdash; this preserves the placeholder's height
contract (text dimensions still drive layout) while removing the visual
text noise that competes with content cards. Set `show_label=True` to
restore the visible label for debugging or alternate consumer contexts.
Padding matches non-placeholder content cards (`p(3)`) so the placeholder's
vertical footprint is consistent with single-line content cards &mdash;
this keeps the slot-count indicator faithful.

In [ ]:
#| export
def render_placeholder_card(
    placeholder_type: Literal["start", "end"],  # Which edge of the list
    show_label: bool = False,                   # Render the "Beginning"/"End" label visibly. Default hides via visibility.invisible (preserves height contract).
) -> Any:  # Placeholder card component
    """Render a placeholder card for viewport edges (fills geometric slot so focus position stays centered)."""
    text = "Beginning" if placeholder_type == "start" else "End"

    text_classes = [font_size.lg, italic, text_tiers.subtle]
    if not show_label:
        text_classes.append(visibility.invisible)

    return Div(
        Div(
            P(
                text,
                cls=combine_classes(*text_classes)
            ),
            cls=combine_classes(card_body, p(3))
        ),
        cls=combine_classes(
            card, "placeholder-card",
            bg_dui.base_100.opacity(50), shadow.none,
            border(2), border_color.transparent
        ),
        data_placeholder_type=placeholder_type
    )

In [ ]:
# Test render_placeholder_card
from fasthtml.common import to_xml

# Default (show_label=False): label text is in the DOM but hidden via
# visibility.invisible — preserves height contract while removing visible
# "Beginning"/"End" text noise from the card stack viewport.
start_card = render_placeholder_card("start")
html = to_xml(start_card)
assert 'data-placeholder-type="start"' in html
assert 'placeholder-card' in html
# Label text still in DOM (height contract preserved)
assert 'Beginning' in html
# Visibility class applied (text invisible)
assert 'invisible' in html

# End variant — same shape, different text content.
end_card = render_placeholder_card("end")
end_html = to_xml(end_card)
assert 'data-placeholder-type="end"' in end_html
assert 'End' in end_html
assert 'invisible' in end_html

# show_label=True restores visible label (no `invisible` class).
labeled = render_placeholder_card("start", show_label=True)
labeled_html = to_xml(labeled)
assert 'Beginning' in labeled_html
assert 'invisible' not in labeled_html, (
    "show_label=True must produce a P tag without visibility.invisible. "
    f"Got: {labeled_html!r}"
)

# Padding standardized to p(3) — matches non-placeholder content cards
# so the placeholder's vertical footprint is consistent.
assert 'p-3' in html
# Regression guard: the historical p(4) padding must not return.
assert 'p-4' not in html, (
    "Placeholder card body padding must be p(3) (matching non-placeholder cards), "
    "not p(4). A regression to p(4) would distort the slot-count indicator at "
    "every card-stack consumer site."
)

print("render_placeholder_card tests passed!")

render_placeholder_card tests passed!


## render_loading_state

Displayed while the card stack is initializing (e.g., fetching items from a service).

In [ ]:
#| export
def render_loading_state(
    ids: CardStackHtmlIds,  # HTML IDs for this card stack instance
    message: str = "Loading...",  # Loading message text
) -> Any:  # Loading component
    """Render loading state with spinner and message."""
    return Div(
        Div(
            Span(cls=combine_classes(loading, loading_styles.spinner, loading_sizes.lg)),
            P(
                message,
                cls=combine_classes(m.t(4), text_tiers.tertiary)
            ),
            cls=combine_classes(
                flex_display, flex_direction.col, items.center, justify.center,
                p(16)
            )
        ),
        id=ids.loading
    )

In [ ]:
# Test render_loading_state
ids = CardStackHtmlIds(prefix="cs0")
loading_el = render_loading_state(ids)
html = to_xml(loading_el)
assert 'id="cs0-loading"' in html
assert "Loading..." in html

# Test custom message
loading_el = render_loading_state(ids, message="Initializing segments...")
html = to_xml(loading_el)
assert "Initializing segments..." in html
print("render_loading_state tests passed!")

render_loading_state tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()